## Data Cleaning Documentation

### My -rough- NOTES reviewing data documentation:

I will be following the suggested steps for data cleaning provide in Chapter 4 of the textbook (Veridical Data Science): 
1. Learn about the data collection process and the problem domain  
2. Load the data 
3. Examine the data and create action items
4. Clean the data 

Step 1: Learn about the data collection process and problem domain 
Notes from reading data documentation: 
- This is an observational cohort study 
- Participants: children less than 18 years old whom have minor head trauma (evaluated in 25 ERs)

- GOAL of the study: create and validate two prediction rules 
    - WANT: these clinical prediction rules to: 
        - "accurately identify children at near-zero risk of clinically important traumatic brain injuries (TBI's) after blunt trauma" 
        - one rule: for children less than 2 years old 
        - another (separate rule): for children 2 years and older 
- More info on the participants: 
    - subjects presented within 24 hours of head trauma (with Glasgow Coma Scale scores: 14-15)
        - domain knowledge from article:  
            - minor head trauma = GCS scores: 14-15 
                - "These children commonly undergo neuroimaging and account for: 40-60% of those with traumatic brain injuries seen on CT"
            - "injuries needing neurosurgery are very UNCOMMON in children with GCS scores of 14-15" 
                - #### Data Cleaning Action item: GCS Total column should only include values 14-15: I already see some differing (e.g., 3, 9): is this a mistake?
- End "result" of the study: 
    - the "validated prediction rules identified children at very low risk of clinically important TBI" : children for whom CT scans can be routinely avoided.

- Inclusion/Exclusion 
    - regardless of GCS score: "children presenting within 24 hours of head trauma were included"
        - #### Data cleaning above: this might be why we see different GCS total scores 
    - EXCLUDED: "children with trivial injury mechanisms" -> defined by "ground-level falls or walking or running into stationary objects, and no signs or symptoms of head trauma other than scalp abrasions and lacerations" 
        - Also excluded: patients with "penetrating trauma, known as brain tumors, pre-existing neurological disorders complicating assessment, or neuroimaging at an outside hospital before transfer"
        - Excluded: if patients had "ventricular shunts or bleeding disorders" 

- #### Data Collection Process: 
    - "Trained site investigators" and other ER physicians recorded: 
        - patient history, injury mechanism, and symptoms and signs on a "standardized data form" BEFORE knowing the imaging results (if imaging was done)
            - #### Data Cleaning action item: be sure that this observational data matches whom is supposed to be excluded (as described above) and included. 
    
- Outcomes: 
    - ciTBI: clinically important TBI 
        - defined as "death from TBI, neurosurgery, intubation for > 24 hours for TBI, or hospital admission of two or more nights for the TBI in association with TBI on CT" 
            - this was defined a priori = means "before looking a the data"
        - Defined this way to: "exclude brief intubations for imaging or overnight admission for minor CT findings" 
    - "hospitalizations for social reasons were not included" 

    - TBI on CT defined by: 
        - "intracranial hemorrhage or contusion, cerebra edema, traumatic infarction, diffuse axonal injury, shearing injury, sigmoid sinus thrombosis, midline shift of intracranial contents or signs of brain herniation, diastasis of the skull, pneumocephalus, or skull fracture depressed by at least the width of the skull table" 

#### Data Cleaning possible action: be sure data aligned with the descriptions of these defined terms above
    
- Follow-up Procedures: 
    - at ER (emergency department) physician discretion: patients were admitted to hospital
        - records of admitted patients: "reviewed by research coordinators and site investigators to assess CT results and presence of ciTBIs" 
    
    - To identify any missed TBIs: 
        - "research coordinators did standardized telephone surveys of guardians of patients discharged from ER (7-90 days) after the ER visit."
        - "Medical records and imaging results were obtained if a missed traumatic brain injury was suggested at follow-up"
    
    - "If a ciTBI was identified -> patient's outcome was classified accordingly 
    - If unable to contact patient guardian -> reviewed medical record, ER process improvement records, and county morgue records (to ensure that no discharged patient was subsequently diagnosed with ciTBI)"



    
    
        
            

#### Above with be shortened/streamlined later 
- This is all we need for now... before loading in the data and looking at it

#### Suggested Questions to pose/answer during data documentation review (directly from Ch4 Veridical Data Science)
1. What does each variable measure? What real-world quantity is each variable supposed to be capturing?
2. How was the data collected? Mentally visualize the real-world data collection procedure. How was each variable physically measured?
3. What are the observational units? The observational units correspond to the entities on which the measurements are collected, such as people, countries, or years.
4. Is the data relevant to my project? Take a moment to verify that the data that you have is indeed relevant to the question that you are asking.
5. What questions do I have, and what assumptions am I making? Make an effort to physically write down and answer any questions that arise and check any assumptions that you make. This requires a certain level of awareness of your own thought process while you examine the background information and data.

##### From the reading, during cleaning look for: 
- Invalid or inconsistent values
- Improperly formatted missing values
- Nonstandard data format
- Messy column names
- Improper variable types
- Incomplete data

# Loading in the Data (inital visual exploration)

#### NOTE: will need to change the file path to work when its not on my computer

In [ ]:
# Investigate what filepath to the data will be
import os
os.getcwd()


In [ ]:
# Load in the data as a pandas dataframe 
import pandas as pd 
tbi_data_df = pd.read_csv('/Users/maddiemac/Desktop/STAT 214/stat-214/lab1/data/TBI PUD 10-08-2013.csv')


In [ ]:
# Take a look at the head and tail of the data 
tbi_data_df.head()

In [ ]:
tbi_data_df.tail()

In [ ]:
# Investigate shape of the data 
tbi_data_df.shape
print(f"This dataframe has {tbi_data_df.shape[0]} rows and {tbi_data_df.shape[1]} columns")

In [ ]:
# Summarize the number of missing values per column 
    # use pd.set_option to adjust display setting 

pd.set_option('display.max_rows', None)
missing_summary = tbi_data_df.isnull().sum()
print(missing_summary)

#### One of the first things I notice is the non-human readable column names - How do I fix this in a function and not 1-by-1?
- Creating a csv file for which I can add in new column names then switch them all at the same time

In [ ]:
# Export current column names for review 
pd.DataFrame(tbi_data_df.columns, columns=['original']).to_csv('column_reference.csv', index=False)


#### What I notice from the columns as I go through documentation and chose better names: 

- have lots of yes/no variables such as: 
    - Amnesia_verb: Does the patient have amnesia of the event yes/no? 
        - which is coding which 0 = no and 1 = yes? 
        - it appears that 91 is used for missing values (judgment call: yes/no is not gonna be 91)
    - other variables with missing values 91: 
        - HA_verb 

- missing values as 92 (e.g., in colums:)
    - LocLen
    - SeizOccur 
    - SeizLen 

- summary: some columns use 92 for missing values 
    - important to note that 0 is meaningful for some of the columns and should not be treated as missing values 
    - 91 (preverbal) and 90 (other) have meanings for some variables 


- Additionally important to notice: some variables are directly related to one another 
    - e.g., has-variable (yes/no) : related to duration 
        - these should align: i.e., if the answer is no then the duration should be 0
    - variables I've noticed like this: 
        - LOCSeparate: has_loss_of_consciousness_history
        - LocLen: loss_of_consciousness_duration

        - Seiz: has_posttraumatic_seizure
        - SeizOccur: seizure_timing 
        - SeizLen: posttraumatic_seizure_duration

    - Another type of relationship, if no then the second variable should not have a value 
        - SFxPalp: has_palpable_skull_fracture
        - SFxPalpDepress: skull_fracture_depressed 

- GCS total score: recall from documentation the inclusion was participants 14-15 score 
    - there are two groups (1 or 2), 1 being outside 14-15 range, 2 being 14-15 range 
    - does being group 1 align with the inclusion/exclusion principles? 

- AMS: altered mental status yes/no 
    - The original AMS variable must be associated with GCS score < 15 to have a valid input 
    - all the other AMS variables: 
        - at least some should be yes if GCS scores outside 14-15 are included in this dataset by the inclusion/exclusion rules
        - AMSOther variable: also a yes or no? 
- Same thing with Clav(trauma_evidence) and NeuroD + types.... etc. 




#### Finished CSV with new column names: new_col_names.csv

- using python, lets apply the new column names to the dataset

In [ ]:
# Read in cleaned column names
name_mapping = pd.read_csv('new_col_names.csv')

# Check what the columns are actually called
print(name_mapping.columns)

# Create a dictionary mapping old to new names (based on header of new_col_names.csv)
rename_dict = dict(zip(name_mapping['original'], name_mapping['new_names']))

# Rename the columns
tbi_data_df.rename(columns=rename_dict, inplace=True)

In [ ]:
# Visualize the new dataframe (confirm updated column names)
tbi_data_df.head()

## Next Data Cleaning Action: Handling Missing Values (part 1)
- First step: according to the textbook we shouldn't use arbitrary numbers (e.g., 90, 91, 92, etc.) to represent missing values 
- From ch4 VDS: "Rather than aiming to explicitly remove or impute missing values, our goal at the data cleaning stage is to ensure that missing values are appropriately formatted, such as by replacing ambiguous numeric placeholders with explicit missing values" (NaN in python)'


### Based on the documentation 91 is used for a "preverbal/nonverbal" score determined by physicians 
- so this value has meaning and should NOT be treated as missing 

In [ ]:
# First check how many 92 instances we have representing missing values (or not applicable)
print((tbi_data_df == 92).sum().sum())


In [ ]:
# Now replace these (91's and 92's) with the appropriate NaN value
    # also replace actual missing (empty cell) values
import numpy as np 
tbi_data_df.replace([92, ], np.nan, inplace=True)

In [ ]:
# Verify all 91's and 92's have been replaced successfully 
print((tbi_data_df == 92).sum().sum())

In [ ]:
# Visualize data again to see NaN's successfully inputted
tbi_data_df.head()

#### Question: are actual missing values (empty cells) already treated as NaN?
- result of the cell below suggests YES : It appears that empty cells are 'nan' in python (under the hood)

In [ ]:
# Run diagnostic, see what these empty cells are treated as in Python 

# Pick a column we know has "empty" cells
col_name = 'has_event_amnesia'  
print(tbi_data_df[col_name].dtype)
print(tbi_data_df[col_name].unique()[:20])  # First 20 unique values

In [ ]:
# Check for actual missing values (empty cells)
# print(tbi_data_df.isnull().sum().sum())
# tbi_data_df = tbi_data_df.replace('', np.nan)

#### LAST NOTE: Keep the following code in mind for later: when reading in the data (help for function)

In [ ]:
# Tell what to interpret as missing when loading in the data 
    # This will be helpful when making data cleaning into a function 

# Keep this code in mind to use for later 

# When loading the data, tell pandas what to treat as NaN
# tbi_data_df = pd.read_csv('/Users/maddiemac/Desktop/STAT 214/stat-214/lab1/data/TBI PUD 10-08-2013.csv', na_values=[91, 92, '', ' '])

# Next Action Item: Look for invalid/inconsistent values 

#### NOTE as discussed above: a lot of variables are using dummy variables for yes/no categories (0,1) : this must be paid attention to so that 0 is never treated as "missing" in these situations
- Are there other "0's" based on the column they are in that SHOULD be considered "missing value" (i.e., zero doesn't have a meaning)

#### Notice: Patient number is a "label" ... should this be float64 or is it more appropriate to be a str?

### Action Item: start looking at the related columns 
- Looking for incorrect or inconsistent values 

Columns that have one or more related to them
- has_loss_of_consciousness 
- has_posttraumatic_seizure
- has_headache_at_ED
- has_vomiting_post_injury 
- has_altered_mental_status 
- has_palpable_skull_fracture
- has_basilar_skull_fracture
- has_hematomas_or_swellings
- has_trauma_above_clavicles 
- has_neuro_deficit 
- has_other_substantial_injury
- head_imaging_ordered 
    - ct_primary_....
- ct_sedation_given 
    - variables for reason...
- patient age in months = age in years = appropriate group (under 2 or not)
- head_ct_performed & head_ct_performed_in_ed 
    - ct_shows_tbi 
        - ct_finding....
        - ct_extra_finding....
        - death_due_to_tbi (should always be no if ct_shows_tbi was no)

- gcs_total_score: should = the sum of the specific components 
    - gcs_total_score = sum (gcs_eye_score + gcs_verbal_score + gcs_motor_score)
    - gcs_category: based on what the gcs_total_score is 




#### <span style="color: red;"> Starting with a simple check (correct gcs_total_score)</span>

In [ ]:
# Calculate what the total SHOULD be
calculated_total = tbi_data_df['gcs_eye_score'] + tbi_data_df['gcs_verbal_score'] + tbi_data_df['gcs_motor_score']

# Compare to the actual total
inconsistent = tbi_data_df['gcs_total_score'] != calculated_total

# Count how many rows are inconsistent
print(f"Number of inconsistent rows: {inconsistent.sum()}")

# View the inconsistent rows
print(tbi_data_df[inconsistent][['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

In [ ]:
# Find rows where all GCS components AND total are NaN but category is NOT NaN
missing_components_but_has_total = tbi_data_df[
    tbi_data_df['gcs_eye_score'].isna() & 
    tbi_data_df['gcs_verbal_score'].isna() & 
    tbi_data_df['gcs_motor_score'].isna() & 
    tbi_data_df['gcs_total_score'].notna()]

In [ ]:

print(f"Rows with missing components but has category: {len(missing_components_but_has_total)}")

In [ ]:
print(missing_components_but_has_total['gcs_total_score'].value_counts())


#### Possible Judgement call: for GCS scores of 15, missing counts should be fine 
- vs not 15 this might be a data cleaning error: impute the missing values to be the proper sum? 

### <span style="color: red;">This needs to be investigated BEFORE changing the types (otherwise we can't compute a sum of categorical variable types</span>


#### validating my judgment call for replacing missing values to be the sum (if its just one component missing)


- Check for complete rows if the sum of the three components = total gcs score 
    - If yes, that means only these three components are taken into account for the sum = total
    - If not, there might be other factors (unmeasured) contributing to the sum and thus we should leave the missing components alone and not assume any scores 

In [ ]:
# Calculate what the total SHOULD be
calculated_total = tbi_data_df['gcs_eye_score'] + tbi_data_df['gcs_verbal_score'] + tbi_data_df['gcs_motor_score']

# Compare to the actual total
correct_total = tbi_data_df['gcs_total_score'] == calculated_total

# Count how many rows are correct (sum of three components = total)
print(f"Number of consistent rows: {correct_total.sum()}")

# View the consistent rows
print(tbi_data_df[correct_total][['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

### It is clear by the 42055 rows that follow sum of three components = total score for gcs that these are the only factors contributing to the sum 
    - therefore the judgment call idea above is supported 
        - if one component missing (impute with correct value based on the total score and other components)

- <span style="color: red;"> Alternative judgment call could be to leave these missing values alone </span>


In [ ]:
# Find rows where total and at least 2 components have values
can_impute = tbi_data_df[
    tbi_data_df['gcs_total_score'].notna() &
    (
        (tbi_data_df['gcs_eye_score'].notna() & tbi_data_df['gcs_verbal_score'].notna()) |
        (tbi_data_df['gcs_eye_score'].notna() & tbi_data_df['gcs_motor_score'].notna()) |
        (tbi_data_df['gcs_verbal_score'].notna() & tbi_data_df['gcs_motor_score'].notna())
    ) &
    (
        tbi_data_df['gcs_eye_score'].isna() |
        tbi_data_df['gcs_verbal_score'].isna() |
        tbi_data_df['gcs_motor_score'].isna()
    )
]

print(f"Rows where we can impute a missing component: {len(can_impute)}")
print(can_impute[['gcs_total_score', 'gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score']])

In [ ]:
# Now impute the missing values 

components = ['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score']

for idx in can_impute.index:
    row = tbi_data_df.loc[idx, components + ['gcs_total_score']]
    missing_col = [col for col in components if pd.isna(row[col])][0]
    present_cols = [col for col in components if pd.notna(row[col])]
    tbi_data_df.loc[idx, missing_col] = row['gcs_total_score'] - row[present_cols].sum()

# Verify
print(tbi_data_df.loc[can_impute.index, ['gcs_total_score', 'gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score']].head(10))

In [ ]:
# sanity check (re-run the filter and should now be 0)

# Find rows where total and at least 2 components have values
can_impute = tbi_data_df[
    tbi_data_df['gcs_total_score'].notna() &
    (
        (tbi_data_df['gcs_eye_score'].notna() & tbi_data_df['gcs_verbal_score'].notna()) |
        (tbi_data_df['gcs_eye_score'].notna() & tbi_data_df['gcs_motor_score'].notna()) |
        (tbi_data_df['gcs_verbal_score'].notna() & tbi_data_df['gcs_motor_score'].notna())
    ) &
    (
        tbi_data_df['gcs_eye_score'].isna() |
        tbi_data_df['gcs_verbal_score'].isna() |
        tbi_data_df['gcs_motor_score'].isna()
    )
]

print(f"Rows where we can impute a missing component: {len(can_impute)}")


# Now with this judgment call applied - reinvestigate inconsistent rows 
    - Number of inconsistent rows BEFORE = 1344 
    - After = 1285
        - What about these rows is inconsistent 
            - missing component scores is (2 or more missing component scores is okay
        
        - What we would need to fix is if all component scores have a value but their sum does not equal the total 
            - judgment call (same reasoning) we should fix the gcs_total 



In [ ]:
# Calculate what the total SHOULD be
calculated_total = tbi_data_df['gcs_eye_score'] + tbi_data_df['gcs_verbal_score'] + tbi_data_df['gcs_motor_score']

# Only look at rows where ALL components are present
all_components_present = (
    tbi_data_df['gcs_eye_score'].notna() & 
    tbi_data_df['gcs_verbal_score'].notna() & 
    tbi_data_df['gcs_motor_score'].notna()
)

# Compare to the actual total
inconsistent = all_components_present & (tbi_data_df['gcs_total_score'] != calculated_total)

print(f"Number of inconsistent rows: {inconsistent.sum()}")
print(tbi_data_df[inconsistent][['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

#### Take action: replace the incorrect (probably recording error) gcs_total_score with the correct sum of the three components (same judgment call support)

In [ ]:
# fix the total score for these rows 

tbi_data_df.loc[inconsistent, 'gcs_total_score'] = (
    tbi_data_df.loc[inconsistent, 'gcs_eye_score'] + 
    tbi_data_df.loc[inconsistent, 'gcs_verbal_score'] + 
    tbi_data_df.loc[inconsistent, 'gcs_motor_score']
)
tbi_data_df.loc[inconsistent, 'gcs_total_score'] = calculated_total[inconsistent]

In [ ]:
# sanity check (re run the filtering code) : output should be 0 rows 

# Calculate what the total SHOULD be
calculated_total = tbi_data_df['gcs_eye_score'] + tbi_data_df['gcs_verbal_score'] + tbi_data_df['gcs_motor_score']

# Only look at rows where ALL components are present
all_components_present = (
    tbi_data_df['gcs_eye_score'].notna() & 
    tbi_data_df['gcs_verbal_score'].notna() & 
    tbi_data_df['gcs_motor_score'].notna()
)

# Compare to the actual total
inconsistent = all_components_present & (tbi_data_df['gcs_total_score'] != calculated_total)

print(f"Number of inconsistent rows: {inconsistent.sum()}")
print(tbi_data_df[inconsistent][['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

In [ ]:
# final sanity check, reprint the "inconsistent" rows and now the sum should match up 
print(tbi_data_df.loc[[12931, 24761], 
    ['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']
])

#### now that we've made some judgment calls on this, check for proper categories of gcs score

    - group 1 = total score of 3-13 
    - group 2 = total score of 14-15 

In [ ]:
# investigate 

# Find rows where gcs_category is inconsistent with gcs_total_score
invalid_gcs_category = tbi_data_df[
    tbi_data_df['gcs_total_score'].notna() &
    tbi_data_df['gcs_category'].notna() &
    (
        ((tbi_data_df['gcs_total_score'] <= 13) & (tbi_data_df['gcs_category'] != 1)) |
        ((tbi_data_df['gcs_total_score'] >= 14) & (tbi_data_df['gcs_category'] != 2))
    )
]

print(f"Inconsistent gcs_category rows: {len(invalid_gcs_category)}")
print(invalid_gcs_category[['gcs_total_score', 'gcs_category']])

# ALL GOOD! 

#### <span style="color: red;">Lets now explore relation between age in months = age in years</span>

In [ ]:
invalid_age = tbi_data_df[
    tbi_data_df['patient_age_months'].notna() &
    (np.floor(tbi_data_df['patient_age_months'] / 12) != tbi_data_df['patient_age_years'])
]

print(f"Invalid rows: {len(invalid_age)}")
print(invalid_age[['patient_age_months', 'patient_age_years']].head(10))

#### check in reverse mathematically to be sure

In [ ]:
invalid_age_reverse = tbi_data_df[
    tbi_data_df['patient_age_years'].notna() &
    (tbi_data_df['patient_age_months'] < tbi_data_df['patient_age_years'] * 12) |
    (tbi_data_df['patient_age_months'] >= (tbi_data_df['patient_age_years'] + 1) * 12)
]

print(f"Invalid rows (reverse check): {len(invalid_age_reverse)}")
print(invalid_age_reverse[['patient_age_months', 'patient_age_years']].head(10))

#### finally check that ages are put into the appropriate <2  or >2 group 

In [ ]:
invalid_age_category = tbi_data_df[
    tbi_data_df['patient_age_years'].notna() &
    tbi_data_df['patient_age_under_2yr'].notna() &
    (
        # If age < 2, category should be 1
        ((tbi_data_df['patient_age_years'] < 2) & (tbi_data_df['patient_age_under_2yr'] != 1)) |
        # If age >= 2, category should be 2
        ((tbi_data_df['patient_age_years'] >= 2) & (tbi_data_df['patient_age_under_2yr'] != 2))
    )
]

print(f"Invalid rows (age years vs age category inconsistent): {len(invalid_age_category)}")
print(invalid_age_category[['patient_age_years', 'patient_age_months', 'patient_age_under_2yr']].head(10))

## Investigating related columns (dummy variables)

#### First one: has_loss_of_consciousness_history 
- if the answer is no (=0), then the related loss_of_consciousness_duration column should be NaN

- if the answer is yes (=1) or suspected (=2), then missing values are okay (actually missing)

In [ ]:
# Find rows where patient has NO LOC but still has a duration recorded
invalid = tbi_data_df[
    (tbi_data_df['has_loss_of_consciousness_history'] == 0) & 
    (tbi_data_df['loss_of_consciousness_duration'].notna())
]

print(f"Invalid rows (no LOC but has duration): {len(invalid)}")

#### Next: has_posttraumatic_seizure and seizure_timing and seizure_duration

In [ ]:
# Find rows where patient has NO seizure but still has a timing recorded
invalid_seiz = tbi_data_df[
    (tbi_data_df['has_posttraumatic_seizure'] == 0) & 
    (tbi_data_df['seizure_timing'].notna())
]

print(f"Invalid rows (no seizure but has timing): {len(invalid_seiz)}")

In [ ]:
# Find rows where patient has NO seizure but still has a duration recorded
invalid_seiz = tbi_data_df[
    (tbi_data_df['has_posttraumatic_seizure'] == 0) & 
    (tbi_data_df['posttraumatic_seizure_duration'].notna())
]

print(f"Invalid rows (no seizure but has timing): {len(invalid_seiz)}")

#### try to check multiple at once 
- has_headache_at_ED 
    - headache_severity 
    - headache_start_time

In [ ]:
# Find rows where patient has NO headache but has severity OR start time recorded
invalid_headache = tbi_data_df[
    (tbi_data_df['has_headache_at_ED'] == 0) & 
    (
        tbi_data_df['headache_severity'].notna() | 
        tbi_data_df['headache_start_time'].notna()
    )
]

print(f"Invalid rows (no headache but has severity or start time): {len(invalid_headache)}")

In [ ]:
# Next up for has_vomiting_post_injury
    # check when main varibale is 0 or NaN 

invalid_headache = tbi_data_df[
    (
        (tbi_data_df['has_headache_at_ED'] == 0) | 
        (tbi_data_df['has_headache_at_ED'].isna())
    ) & 
    (
        tbi_data_df['headache_severity'].notna() | 
        tbi_data_df['headache_start_time'].notna()
    )
]

print(f"Invalid rows (no/missing headache but has severity or start time): {len(invalid_headache)}")

# MAKING A FUNCTION OUT OF THIS ACTION: 

In [ ]:
# Define the main variable and its related columns
conditional_checks = {
    'has_loss_of_consciousness_history': ['loss_of_consciousness_duration'],
    'has_posttraumatic_seizure': ['seizure_timing', 'posttraumatic_seizure_duration'],
    'has_headache_at_ED': ['headache_severity', 'headache_start_time'],
    'has_vomiting_post_injury': ['number_of_vomiting_episodes', 'vomiting_start_time', 'vomiting_last_time'],
    'has_altered_mental_status': ['ams_agitated', 'ams_sleepy', 'ams_slow_to_respond','ams_repetitive_questions', 'ams_other'], 
    'has_palpable_skull_fracture': ['skull_fracture_depressed'],
    'has_basilar_skull_fracture_signs': ['has_basilar_hemotympanum', 'has_basilar_csf_otorrhea', 'has_basilar_raccoon_eyes', 'has_basilar_battles_sign', 'has_basilar_csf_rhinorrhea'],
    'has_hematomas_or_swellings': ['hemotomas_or_swellings_location', 'largest_hemotoma_or_swelling_size'], 
    'has_trauma_above_clavicles': ['has_trauma_face', 'has_trauma_neck', 'has_trauma_scalp_frontal', 'has_trauma_scalp_occipital', 'has_trauma_scalp_parietal', 'has_trauma_scalp_temporal'],
    'has_neuro_deficit': ['has_neuro_deficit_motor', 'has_neuro_deficit_sensory', 'has_neuro_deficit_cranial_nerve', 'has_neuro_deficit_reflexes', 'has_neuro_deficit_other'], 
    'has_other_substantial_injury_non_head': ['has_osi_extremity', 'has_osi_laceration_requiring_operation', 'has_osi_cervical_spine', 'has_osi_chest_back_flank', 'has_osi_abdominal', 'has_osi_pelvis', 'has_osi_other'], 
    'ct_head_imaging_ordered': ['ct_primary_indication_young_age', 'ct_primary_indication_amnesia', 'ct_primary_indication_altered_mental_status', 'ct_primary_indication_skull_fracture', 'ct_primary_indication_headache', 'ct_primary_indication_scalp_hematoma', 'ct_primary_indication_loss_of_consciousness', 'ct_primary_indication_injury_mechanism', 'ct_primary_indication_neuro_deficit', 'ct_primary_indication_md_request', 'ct_primary_indication_parental_request', 'ct_primary_indication_trauma_team_request', 'ct_primary_indication_seizure', 'ct_primary_indication_vomiting', 'ct_primary_indication_skull_fracture_on_xray', 'ct_primary_indication_other'],
    'ct_sedation_given': ['ct_sedation_reason_agitation', 'ct_sedation_reason_young_age', 'ct_sedation_reason_technician_request', 'ct_sedation_reason_other'],
    'head_ct_performed': ['ct_shows_tbi'], 
    'head_ct_performed_in_ed': ['ct_shows_tbi'], 
    'ct_shows_tbi': ['ct_finding_cerebellar_hemorrhage', 'ct_finding_cerebral_contusion', 'ct_finding_cerebral_edema', 'ct_finding_cerebral_hemorrhage', 'ct_finding_skull_diastasis', 'ct_finding_epidural_hematoma', 'ct_finding_extra_axial_hematoma', 'ct_finding_intraventricular_hemorrhage', 'ct_finding_midline_shift', 'ct_finding_pneumocephalus', 'ct_finding_skull_fracture', 'ct_finding_subarachnoid_hemorrhage', 'ct_finding_subdural_hematoma', 'ct_finding_traumatic_infarction', 'ct_extra_finding_diffuse_axonal_injury', 'ct_extra_finding_herniation', 'ct_extra_finding_shear_injury', 'ct_extra_finding_sigmoid_sinus_thrombosis'], 
    'has_clinically_important_tbi': ['death_due_to_tbi']

}

def check_conditional_validity(df, checks):
    for main_col, related_cols in checks.items():
        # Main variable is 0 or NaN
        main_invalid = (df[main_col] == 0) | (df[main_col].isna())
        
        # Any related column has a meaningful value (not NaN and not 0)
        related_has_value = (df[related_cols[0]].notna() & (df[related_cols[0]] != 0))
        for col in related_cols[1:]:
            related_has_value = related_has_value | (df[col].notna() & (df[col] != 0))
        
        # Find invalid rows
        invalid = df[main_invalid & related_has_value]
        
        print(f"Main: {main_col}")
        print(f"  Related: {related_cols}")
        print(f"  Invalid rows: {len(invalid)}")
        print("---")

check_conditional_validity(tbi_data_df, conditional_checks)

#### Two curious results 
- Main: head_ct_performed_in_ed
    - Related: ['ct_shows_tbi']
    - Invalid rows: 8
AND
- Main: ct_shows_tbi
    - Related: ['ct_finding_cerebellar_hemorrhage', 'ct_finding_cerebral_contusion', 'ct_finding_cerebral_edema', 'ct_finding_cerebral_hemorrhage', 'ct_finding_skull_diastasis', 'ct_finding_epidural_hematoma', 'ct_finding_extra_axial_hematoma', 'ct_finding_intraventricular_hemorrhage', 'ct_finding_midline_shift', 'ct_finding_pneumocephalus', 'ct_finding_skull_fracture', 'ct_finding_subarachnoid_hemorrhage', 'ct_finding_subdural_hematoma', 'ct_finding_traumatic_infarction', 'ct_extra_finding_diffuse_axonal_injury', 'ct_extra_finding_herniation', 'ct_extra_finding_shear_injury', 'ct_extra_finding_sigmoid_sinus_thrombosis']
    - Invalid rows: 500

#### explore the first one
- if head ct not performed in ed 
    - could be yes to the basic: head_ct_performed

    - head_ct_performed_in_ed = 0 doesn't necessarily mean no CT was done, it just means it wasn't done in the ED specifically
        - head_ct_performed is the overall "was any CT done" variable

In [ ]:
# Rerun but save the invalid rows and inspect them
invalid_ct = tbi_data_df[
    (
        (tbi_data_df['head_ct_performed_in_ed'] == 0) | 
        (tbi_data_df['head_ct_performed_in_ed'].isna())
    ) & 
    tbi_data_df['ct_shows_tbi'].notna()
]

# See the actual values in both columns
print(invalid_ct[['head_ct_performed_in_ed', 'ct_shows_tbi']].value_counts())

# Investigate with head_ct_performed
print(invalid_ct[['head_ct_performed', 'head_ct_performed_in_ed', 'ct_shows_tbi']].value_counts())

#### new code: more readable output: actually looks fine
- first inconsistency flagged is OKAY 

In [ ]:
# Flag rows that are truly invalid:
# head_ct_performed_in_ed = 0 AND ct_shows_tbi = 1 AND head_ct_performed != 1
truly_invalid_ct = tbi_data_df[
    (tbi_data_df['head_ct_performed_in_ed'] == 0) &
    (tbi_data_df['ct_shows_tbi'] == 1) &
    (tbi_data_df['head_ct_performed'] != 1)
]

print(f"Truly invalid rows: {len(truly_invalid_ct)}")
print(truly_invalid_ct[['head_ct_performed', 'head_ct_performed_in_ed', 'ct_shows_tbi']])

In [ ]:
print(invalid_ct[['head_ct_performed', 'head_ct_performed_in_ed', 'ct_shows_tbi']])

### NOW check through the second inconsistency found

- Main: ct_shows_tbi
    - Related: ['ct_finding_cerebellar_hemorrhage', 'ct_finding_cerebral_contusion', 'ct_finding_cerebral_edema', 'ct_finding_cerebral_hemorrhage', 'ct_finding_skull_diastasis', 'ct_finding_epidural_hematoma', 'ct_finding_extra_axial_hematoma', 'ct_finding_intraventricular_hemorrhage', 'ct_finding_midline_shift', 'ct_finding_pneumocephalus', 'ct_finding_skull_fracture', 'ct_finding_subarachnoid_hemorrhage', 'ct_finding_subdural_hematoma', 'ct_finding_traumatic_infarction', 'ct_extra_finding_diffuse_axonal_injury', 'ct_extra_finding_herniation', 'ct_extra_finding_shear_injury', 'ct_extra_finding_sigmoid_sinus_thrombosis']
    - Invalid rows: 500

In [ ]:
ct_finding_cols = [
    'ct_finding_cerebellar_hemorrhage',
    'ct_finding_cerebral_contusion',
    'ct_finding_cerebral_edema',
    'ct_finding_cerebral_hemorrhage',
    'ct_finding_skull_diastasis',
    'ct_finding_epidural_hematoma',
    'ct_finding_extra_axial_hematoma',
    'ct_finding_intraventricular_hemorrhage',
    'ct_finding_midline_shift',
    'ct_finding_pneumocephalus',
    'ct_finding_skull_fracture',
    'ct_finding_subarachnoid_hemorrhage',
    'ct_finding_subdural_hematoma',
    'ct_finding_traumatic_infarction',
    'ct_extra_finding_diffuse_axonal_injury',
    'ct_extra_finding_herniation',
    'ct_extra_finding_shear_injury',
    'ct_extra_finding_sigmoid_sinus_thrombosis'
]

# Only show rows where at least one finding is actually 1
related_condition = (tbi_data_df[ct_finding_cols] == 1).any(axis=1)

invalid_ct_tbi = tbi_data_df[
    (
        (tbi_data_df['ct_shows_tbi'] == 0) |
        (tbi_data_df['ct_shows_tbi'].isna())
    ) &
    related_condition
]

print(f"Invalid rows: {len(invalid_ct_tbi)}")

# Only show columns where at least one value is 1
cols_with_issues = [col for col in ct_finding_cols if (invalid_ct_tbi[col] == 1).any()]
print(invalid_ct_tbi[['ct_shows_tbi'] + cols_with_issues])

According to documentation: 
- Non-depressed skull fracture → does NOT count as TBI on CT
- Depressed skull fracture (by at least the width of the skull) → DOES count as TBI on CT

So:
- ct_finding_skull_fracture = 1 AND skull_fracture_depressed = 0 → ct_shows_tbi can stay 0 
- ct_finding_skull_fracture = 1 AND skull_fracture_depressed = 1 → ct_shows_tbi should be 1 

Let's check this out with code

In [ ]:
print(invalid_ct_tbi[['ct_shows_tbi', 'ct_finding_skull_fracture', 'skull_fracture_depressed']].value_counts())

In [ ]:
print(invalid_ct_tbi[
    invalid_ct_tbi['skull_fracture_depressed'] == 1
][['ct_shows_tbi', 'ct_finding_skull_fracture', 'skull_fracture_depressed']])

## SO does this mean we should impute the values here for ct_shows_tbi to be = 1 based on domain knowledge?
- Yes! According to the article's Panel 2, a depressed skull fracture absolutely counts as TBI on CT. So these 9 rows where ct_finding_skull_fracture = 1 AND skull_fracture_depressed = 1 but ct_shows_tbi = 0 are genuine data entry errors.

## <span style="color: red;">IMPUTE CODE: make sure this is okay to do</span>


In [ ]:
# Get the index of rows where skull fracture is depressed but ct_shows_tbi is incorrectly 0
depressed_fracture_rows = invalid_ct_tbi[
    invalid_ct_tbi['skull_fracture_depressed'] == 1
].index

# Impute ct_shows_tbi to 1 for these rows
tbi_data_df.loc[depressed_fracture_rows, 'ct_shows_tbi'] = 1

# Verify the fix
print(f"Rows corrected: {len(depressed_fracture_rows)}")
print(tbi_data_df.loc[depressed_fracture_rows, ['ct_shows_tbi', 'ct_finding_skull_fracture', 'skull_fracture_depressed']])

## Lets move onto investigating appropriate column variable TYPES in Python

In [ ]:
# look at the data types of each column 
tbi_data_df.dtypes

### Action item: for the columns that include categorical data, meaning the values represent categories, we should make the type different 
- in pandas (what we load the data in with) we can use the 'category' type 
- we can do this all at once with a for loop 

### <span style="color: red;"> NOTE that for preprocessing task: might need to convert the category types back to a numeric type for apply certain models/algorithms</span>
- general code to do so: df["column_name_new"] = df["column_name_converted"].map({"No": 0, "Yes": 1})



### General Action step 
    - categorical_columns: when numeric values represent categories 
    - binary_cat_columns: same as above but when the column only takes 0 or 1 (with possible 90,91 representing PV and other)
    - integer_columns: numeric values that don't make sense to be measured with a decimal value (don't need to be float)

NOTE: 92 is "not applicable": judgment call to treat theses as NaN

In [ ]:
categorical_cols = [
    'physician_employment_type',       # 1-5 categories
    'physician_certification',         # 1-4 + 90 (other)
    'injury_mechanism',                # 1-12 + 90 (other)
    'injury_mechanism_severity',       # 1=Low, 2=Moderate, 3=High
    'loss_of_consciousness_duration',  # 1-4 ordered categories
    'seizure_timing',                  # 1-3 ordered categories
    'posttraumatic_seizure_duration',  # 1-4 ordered categories
    'headache_severity',               # 1-3 ordered categories
    'headache_start_time',             # 1-4 ordered categories
    'number_of_vomiting_episodes',     # 1-3 ordered categories
    'vomiting_start_time',             # 1-4 ordered categories
    'vomiting_last_time',              # 1-4 ordered categories
    'gcs_eye_score',                   # 1-4 ordered
    'gcs_verbal_score',                # 1-5 ordered
    'gcs_motor_score',                 # 1-6 ordered
    'gcs_category',                    # group is 1=3-13, group is 2=14-15
    'hemotomas_or_swellings_location', # 1-3 categories
    'largest_hemotoma_or_swelling_size', # 1-3 ordered
    'patient_gender',                  # 1=Male, 2=Female
    'patient_ethnicity',               # 1-2
    'patient_race',                    # 1-5 + 90 (other)
    'ed_discharge_status',             # 1-8 + 90 (other)
    'patient_age_under_2yr',           # 1=<2yr, 2=>=2yr
]

In [ ]:
# All binary yes/no variables (0=No, 1=Yes, with possible NaN or additional values like 90, 91, 2 (mean other or PV))

binary_yes_no_cols = [
    'has_event_amnesia',  # 91 is also a category now (preverbal/nonverbal)
    'has_loss_of_consciousness_history',  # 0/1/2 (2 = suspected)
    'has_posttraumatic_seizure',
    'acting_normal',
    'has_headache_at_ED',  # has 91 too (preverbal/nonverbal)
    'has_vomiting_post_injury',
    'has_dizziness_at_ED',
    'eval_after_intubation',
    'eval_after_paralysis',
    'eval_after_sedated',
    'has_altered_mental_status',
    'ams_agitated',   # 92 = NA 
    'ams_sleepy',   # 92 = NA 
    'ams_slow_to_respond',   # 92 = NA 
    'ams_repetitive_questions',   # 92 = NA 
    'ams_other',   # 92 = NA 
    'has_palpable_skull_fracture',  # 0/1/2 (2=unclear exam)
    'skull_fracture_depressed',   # 92 = NA 
    'has_anterior_fontanelle_bulging',
    'has_basilar_skull_fracture_signs',
    'has_basilar_hemotympanum',   # 92 = NA 
    'has_basilar_csf_otorrhea',   # 92 = NA 
    'has_basilar_raccoon_eyes',   # 92 = NA 
    'has_basilar_battles_sign',   # 92 = NA 
    'has_basilar_csf_rhinorrhea',   # 92 = NA 
    'has_hematomas_or_swellings',
    'has_trauma_above_clavicles',
    'has_trauma_face',   # 92 = NA 
    'has_trauma_neck',   # 92 = NA 
    'has_trauma_scalp_frontal',   # 92 = NA 
    'has_trauma_scalp_occipital',   # 92 = NA 
    'has_trauma_scalp_parietal',   # 92 = NA 
    'has_trauma_scalp_temporal',   # 92 = NA 
    'has_neuro_deficit',
    'has_neuro_deficit_motor',   # 92 = NA 
    'has_neuro_deficit_sensory',   # 92 = NA 
    'has_neuro_deficit_cranial_nerve',   # 92 = NA 
    'has_neuro_deficit_reflexes',   # 92 = NA 
    'has_neuro_deficit_other',   # 92 = NA 
    'has_other_substantial_injury_non_head',
    'has_osi_extremity',   # 92 = NA 
    'has_osi_laceration_requiring_operation',   # 92 = NA 
    'has_osi_cervical_spine',   # 92 = NA 
    'has_osi_chest_back_flank',   # 92 = NA 
    'has_osi_abdominal',   # 92 = NA 
    'has_osi_pelvis',   # 92 = NA 
    'has_osi_other',   # 92 = NA 
    'clinical_suspicion_of_intoxication',
    'ct_head_imaging_ordered',
    'ct_primary_indication_young_age',   # 92 = NA 
    'ct_primary_indication_amnesia',   # 92 = NA 
    'ct_primary_indication_altered_mental_status',   # 92 = NA 
    'ct_primary_indication_skull_fracture',   # 92 = NA 
    'ct_primary_indication_headache',   # 92 = NA 
    'ct_primary_indication_scalp_hematoma',   # 92 = NA 
    'ct_primary_indication_loss_of_consciousness',   # 92 = NA 
    'ct_primary_indication_injury_mechanism',   # 92 = NA 
    'ct_primary_indication_neuro_deficit',   # 92 = NA 
    'ct_primary_indication_md_request',   # 92 = NA 
    'ct_primary_indication_parental_request',   # 92 = NA 
    'ct_primary_indication_trauma_team_request',   # 92 = NA 
    'ct_primary_indication_seizure',   # 92 = NA 
    'ct_primary_indication_vomiting',   # 92 = NA 
    'ct_primary_indication_skull_fracture_on_xray',   # 92 = NA 
    'ct_primary_indication_other',   # 92 = NA 
    'ct_sedation_given',   # 92 = NA 
    'ct_sedation_reason_agitation',   # 92 = NA 
    'ct_sedation_reason_young_age',   # 92 = NA 
    'ct_sedation_reason_technician_request',   # 92 = NA 
    'ct_sedation_reason_other',   # 92 = NA 
    'ed_observation_for_ct_decision',
    'head_ct_performed_in_ed',
    'ct_shows_tbi',   # 92 = NA 
    'ct_finding_cerebellar_hemorrhage',   # 92 = NA 
    'ct_finding_cerebral_contusion',   # 92 = NA 
    'ct_finding_cerebral_edema',   # 92 = NA 
    'ct_finding_cerebral_hemorrhage',   # 92 = NA 
    'ct_finding_skull_diastasis',   # 92 = NA 
    'ct_finding_epidural_hematoma',   # 92 = NA 
    'ct_finding_extra_axial_hematoma',
    'ct_finding_intraventricular_hemorrhage',   # 92 = NA 
    'ct_finding_midline_shift',   # 92 = NA 
    'ct_finding_pneumocephalus',   # 92 = NA 
    'ct_finding_skull_fracture',   # 92 = NA 
    'ct_finding_subarachnoid_hemorrhage',   # 92 = NA 
    'ct_finding_subdural_hematoma',   # 92 = NA 
    'ct_finding_traumatic_infarction',   # 92 = NA 
    'ct_extra_finding_diffuse_axonal_injury',   # 92 = NA 
    'ct_extra_finding_herniation',   # 92 = NA 
    'ct_extra_finding_shear_injury',   # 92 = NA 
    'ct_extra_finding_sigmoid_sinus_thrombosis',   # 92 = NA 
    'death_due_to_tbi',
    'hospitalized_2plus_nights_head_injury',
    'hospitalized_2plus_nights_with_positive_ct',
    'intubated_24plus_hours_head_trauma',
    'neurosurgery_performed',
    'has_clinically_important_tbi',
]

# Convert to category
for col in binary_yes_no_cols:
    tbi_data_df[col] = tbi_data_df[col].astype('category')

In [ ]:
integer_cols = [
    #'patient_number',   # unique numeric identifier
    'patient_age_months',   # continuous numeric
    'gcs_total_score',      # numeric sum 3-15
]

In [ ]:
# conversion all at once 

# Convert to category
for col in categorical_cols:
    tbi_data_df[col] = tbi_data_df[col].astype('category')
for col in binary_yes_no_cols: 
    tbi_data_df[col] = tbi_data_df[col].astype('category')

# Convert to int where possible (only if no NaNs)
for col in integer_cols:
    if tbi_data_df[col].notna().all():
        tbi_data_df[col] = tbi_data_df[col].astype('int64')

In [ ]:
# sanity check 
tbi_data_df.dtypes


In [ ]:
# patient_number conversion didn't work , meaning it must have at least one NaN value.... this is problematic 
    # investigate where the missing patient_numbers are
tbi_data_df[tbi_data_df["patient_number"].isna()].index

#### Action: Impute missing values for patient number: this makes sense and is fine because each patient should have a unique identifier and since we know the identifiers are in ascending order: we can replace the missing value with the correct value that we know because of the numbers surrounding
- Do this with code below

In [ ]:
# Find which numbers are missing from the sequence
existing_nums = tbi_data_df['patient_number'].dropna().astype(int)
min_num = existing_nums.min()
max_num = existing_nums.max()

# Find gaps in the sequence
expected_nums = set(range(int(min_num), int(max_num) + 1))
actual_nums = set(existing_nums)
missing_nums = sorted(expected_nums - actual_nums)

# Assign missing numbers to NaN rows
missing_indices = tbi_data_df[tbi_data_df['patient_number'].isna()].index
for idx, num in zip(missing_indices, missing_nums):
    tbi_data_df.loc[idx, 'patient_number'] = num

# Convert to Int64
tbi_data_df['patient_number'] = tbi_data_df['patient_number'].astype('Int64')

### NOTE: might want to take the last line out, do this code before the conversion of all types, then at patient_number to that larger conversion loop

## <span style="color: red;">This mightve happened because of making 92 = NaN for ALL columns : go back and fix these missing values for all columns EXCEPT FOR patient_number</span>

In [ ]:
# patient_age_months also didn't convert from float to int properly
    # investigate missing values 
tbi_data_df[tbi_data_df["patient_age_months"].isna()].index

In [ ]:
tbi_data_df['patient_age_months'].dtype

### For this judgment call: 
- missing values for age in months should be imputed if and only if their corresponding age in years is 0 
- otherwise the age in months could be anything that falls within that whole year number (so it is better to leave it missing then guess the wrong number)

In [ ]:
# First, impute missing age_months as 0 ONLY when age_years is 0 (newborns)
tbi_data_df.loc[
    (tbi_data_df['patient_age_months'].isna()) & 
    (tbi_data_df['patient_age_years'] == 0),
    'patient_age_months'
] = 0

# Now convert to nullable Int64 type (capital I) which can hold NaN
tbi_data_df['patient_age_months'] = tbi_data_df['patient_age_months'].astype('Int64')

# Verify
print(f"Remaining NaN in patient_age_months: {tbi_data_df['patient_age_months'].isna().sum()}")
print(f"Data type: {tbi_data_df['patient_age_months'].dtype}")

### <span style="color: red;"> Same thing as before, maybe wanna do this first before the full -at at once- conversion</span>
- add a line of code that allows conversion to Int64 (capital I) so that it can hold NaN values for certain columns

In [ ]:
# sanity check that types have been updated successfully now 
tbi_data_df.dtypes

### Next idea - when 0s mean something vs when the value should be missing?
- or do some exploratory analysis to figure out what else needs to be cleaned 
- question : are there other "0's" based on the column they are in that SHOULD be considered "missing value" (i.e., zero doesn't have a meaning)

In [ ]:
# Define columns where 0 has no valid meaning
cols_where_zero_is_invalid = [
    'patient_number',
    'physician_employment_type',
    'physician_certification',
    'injury_mechanism',
    'injury_mechanism_severity',
    'loss_of_consciousness_duration',
    'seizure_timing', 
    'posttraumatic_seizure_duration',
    'headache_severity', 
    'headache_start_time',
    'number_of_vomiting_episodes', 
    'vomiting_start_time', 
    'vomiting_last_time', 
    'gcs_eye_score',
    'gcs_verbal_score', 
    'gcs_motor_score',
    'gcs_category',
    'hemotomas_or_swellings_location', 
    'largest_hemotoma_or_swelling_size', 
    'patient_age_months',  # can't be 0 months old
    'patient_age_under_2yr',
    'patient_gender',
    'patient_ethnicity',
    'patient_race',
    'ed_discharge_status',
]

# Check for invalid 0s
for col in cols_where_zero_is_invalid:
    count_zeros = (tbi_data_df[col] == 0).sum()
    if count_zeros > 0:
        print(f"{col}: {count_zeros} invalid zeros")

#### NOTE: thought about it an patients CAN be zero months old (as long as years age is also 0 and age 2 yrs is 1 (under 2 yrs)) meaning for example they are only weeks only 

In [ ]:
# check how many above are valid (newborns)
# Check the 0 month rows
zero_month_rows = tbi_data_df[tbi_data_df['patient_age_months'] == 0]

print(f"Total rows with 0 months: {len(zero_month_rows)}")
print("\nBreakdown:")
print(f"  patient_age_years == 0: {(zero_month_rows['patient_age_years'] == 0).sum()}")
print(f"  patient_age_under_2yr == 1: {(zero_month_rows['patient_age_under_2yr'] == 1).sum()}")

# Find truly invalid ones (0 months but NOT a newborn)
invalid_zero_months = zero_month_rows[
    (zero_month_rows['patient_age_years'] != 0) | 
    (zero_month_rows['patient_age_under_2yr'] != 1)
]

print(f"\nTruly invalid 0 months: {len(invalid_zero_months)}")
print(invalid_zero_months[['patient_age_months', 'patient_age_years', 'patient_age_under_2yr']])

#### So all of the above are valid

## <span style="color: red;"> MORE CLEANING: Recall from the very beginning from chapter</span>

#### Suggested Questions to pose/answer during data documentation review (directly from Ch4 Veridical Data Science)
1. What does each variable measure? What real-world quantity is each variable supposed to be capturing?
2. How was the data collected? Mentally visualize the real-world data collection procedure. How was each variable physically measured?
3. What are the observational units? The observational units correspond to the entities on which the measurements are collected, such as people, countries, or years.
4. Is the data relevant to my project? Take a moment to verify that the data that you have is indeed relevant to the question that you are asking.
5. What questions do I have, and what assumptions am I making? Make an effort to physically write down and answer any questions that arise and check any assumptions that you make. This requires a certain level of awareness of your own thought process while you examine the background information and data.

##### From the reading, during cleaning look for: 
- Invalid or inconsistent values
- Improperly formatted missing values
- Nonstandard data format
- Messy column names
- Improper variable types
- Incomplete data



# more investigating inconsistent or incorrect data entires based on column relations
## <span style="color: red;">Are there others?</span>

- Worth looking at: 
1. skull_fracture_depressed = 1 → has_palpable_skull_fracture must = 1
   (can't have a depressed fracture without a palpable fracture)

2. hospitalized_2plus_nights_with_positive_ct = 1 → both:
   - hospitalized_2plus_nights_head_injury must = 1
   - ct_shows_tbi must = 1
   (it's literally the intersection of both)

3. head_ct_performed_in_ed = 1 → head_ct_performed must = 1
   (can't have done CT in ED without having done a CT at all)

### 1. can't have depressed fracture without a palpable fracture 

In [ ]:
invalid_depressed_fracture = tbi_data_df[
    (tbi_data_df['skull_fracture_depressed'] == 1) &
    (tbi_data_df['has_palpable_skull_fracture'] != 1)
]

print(f"Invalid rows: {len(invalid_depressed_fracture)}")
print(invalid_depressed_fracture[['skull_fracture_depressed', 'has_palpable_skull_fracture']])

### 2. hospitalized_2plus_nights_with_positive_ct = 1 → both:
   - hospitalized_2plus_nights_head_injury must = 1
   - ct_shows_tbi must = 1
   (it's literally the intersection of both)


In [ ]:
invalid_hospitalized_ct = tbi_data_df[
    (tbi_data_df['hospitalized_2plus_nights_with_positive_ct'] == 1) &
    (
        (tbi_data_df['hospitalized_2plus_nights_head_injury'] != 1) |
        (tbi_data_df['ct_shows_tbi'] != 1)
    )
]

print(f"Invalid rows: {len(invalid_hospitalized_ct)}")
print(invalid_hospitalized_ct[['hospitalized_2plus_nights_with_positive_ct', 'hospitalized_2plus_nights_head_injury', 'ct_shows_tbi']])

### 3. can't have a head CT done in ED without having one done at all 

In [ ]:
invalid_ed_ct = tbi_data_df[
    (tbi_data_df['head_ct_performed_in_ed'] == 1) &
    (tbi_data_df['head_ct_performed'] != 1)
]

print(f"Invalid rows: {len(invalid_ed_ct)}")
print(invalid_ed_ct[['head_ct_performed_in_ed', 'head_ct_performed']])

### ALL GOOD for those three investigations! 

## <span style="color: red;">Anything else to investigate in this manner before moving onto EDA? </span>


### Inconsistent/incorrect values 
- based on the documentation : a lot of categorical variables have ranges to them, lets make sure that there are no values outside the correct range of what can be taken on 

- In code try to specify the range these values can take on and investigate if any rows have entries outside that range 
    - either make missing or impute if appropriate



In [ ]:
# Define valid values for every column based on documentation
valid_values = {
    'physician_employment_type':        [1, 2, 3, 4, 5],
    'physician_certification':          [1, 2, 3, 4, 90],
    'injury_mechanism':                 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 90],
    'injury_mechanism_severity':        [1, 2, 3],
    'has_event_amnesia':                [0, 1, 91],
    'has_loss_of_consciousness_history':[0, 1, 2],
    'loss_of_consciousness_duration':   [1, 2, 3, 4],
    'has_posttraumatic_seizure':        [0, 1],
    'seizure_timing':                   [1, 2, 3],
    'posttraumatic_seizure_duration':   [1, 2, 3, 4],
    'acting_normal':                    [0, 1],
    'has_headache_at_ED':               [0, 1, 91],
    'headache_severity':                [1, 2, 3],
    'headache_start_time':              [1, 2, 3, 4],
    'has_vomiting_post_injury':         [0, 1],
    'number_of_vomiting_episodes':      [1, 2, 3],
    'vomiting_start_time':              [1, 2, 3, 4],
    'vomiting_last_time':               [1, 2, 3],
    'has_dizziness_at_ED':              [0, 1],
    'eval_after_intubation':            [0, 1],
    'eval_after_paralysis':             [0, 1],
    'eval_after_sedated':               [0, 1],
    'gcs_eye_score':                    [1, 2, 3, 4],
    'gcs_verbal_score':                 [1, 2, 3, 4, 5],
    'gcs_motor_score':                  [1, 2, 3, 4, 5, 6],
    'gcs_category':                     [1, 2],
    'has_altered_mental_status':        [0, 1],
    'ams_agitated':                     [0, 1],
    'ams_sleepy':                       [0, 1],
    'ams_slow_to_respond':              [0, 1],
    'ams_repetitive_questions':         [0, 1],
    'ams_other':                        [0, 1],
    'has_palpable_skull_fracture':      [0, 1, 2],
    'skull_fracture_depressed':         [0, 1],
    'has_anterior_fontanelle_bulging':  [0, 1],
    'has_basilar_skull_fracture_signs': [0, 1],
    'has_basilar_hemotympanum':         [0, 1],
    'has_basilar_csf_otorrhea':         [0, 1],
    'has_basilar_raccoon_eyes':         [0, 1],
    'has_basilar_battles_sign':         [0, 1],
    'has_basilar_csf_rhinorrhea':       [0, 1],
    'has_hematomas_or_swellings':       [0, 1],
    'hemotomas_or_swellings_location':  [1, 2, 3],
    'largest_hemotoma_or_swelling_size':[1, 2, 3],
    'has_trauma_above_clavicles':       [0, 1],
    'has_trauma_face':                  [0, 1],
    'has_trauma_neck':                  [0, 1],
    'has_trauma_scalp_frontal':         [0, 1],
    'has_trauma_scalp_occipital':       [0, 1],
    'has_trauma_scalp_parietal':        [0, 1],
    'has_trauma_scalp_temporal':        [0, 1],
    'has_neuro_deficit':                [0, 1],
    'has_neuro_deficit_motor':          [0, 1],
    'has_neuro_deficit_sensory':        [0, 1],
    'has_neuro_deficit_cranial_nerve':  [0, 1],
    'has_neuro_deficit_reflexes':       [0, 1],
    'has_neuro_deficit_other':          [0, 1],
    'has_other_substantial_injury_non_head': [0, 1],
    'has_osi_extremity':                [0, 1],
    'has_osi_laceration_requiring_operation': [0, 1],
    'has_osi_cervical_spine':           [0, 1],
    'has_osi_chest_back_flank':         [0, 1],
    'has_osi_abdominal':                [0, 1],
    'has_osi_pelvis':                   [0, 1],
    'has_osi_other':                    [0, 1],
    'clinical_suspicion_of_intoxication': [0, 1],
    'ct_head_imaging_ordered':          [0, 1],
    'ct_primary_indication_young_age':  [0, 1],
    'ct_primary_indication_amnesia':    [0, 1],
    'ct_primary_indication_altered_mental_status': [0, 1],
    'ct_primary_indication_skull_fracture': [0, 1],
    'ct_primary_indication_headache':   [0, 1],
    'ct_primary_indication_scalp_hematoma': [0, 1],
    'ct_primary_indication_loss_of_consciousness': [0, 1],
    'ct_primary_indication_injury_mechanism': [0, 1],
    'ct_primary_indication_neuro_deficit': [0, 1],
    'ct_primary_indication_md_request': [0, 1],
    'ct_primary_indication_parental_request': [0, 1],
    'ct_primary_indication_trauma_team_request': [0, 1],
    'ct_primary_indication_seizure':    [0, 1],
    'ct_primary_indication_vomiting':   [0, 1],
    'ct_primary_indication_skull_fracture_on_xray': [0, 1],
    'ct_primary_indication_other':      [0, 1],
    'ct_sedation_given':                [0, 1],
    'ct_sedation_reason_agitation':     [0, 1],
    'ct_sedation_reason_young_age':     [0, 1],
    'ct_sedation_reason_technician_request': [0, 1],
    'ct_sedation_reason_other':         [0, 1],
    'patient_age_under_2yr':            [1, 2],
    'patient_gender':                   [1, 2],
    'patient_ethnicity':                [1, 2],
    'patient_race':                     [1, 2, 3, 4, 5, 90],
    'ed_observation_for_ct_decision':   [0, 1],
    'ed_discharge_status':              [1, 2, 3, 4, 5, 6, 7, 8, 90],
    'head_ct_performed':                [0, 1],
    'head_ct_performed_in_ed':          [0, 1],
    'ct_shows_tbi':                     [0, 1],
    'ct_finding_cerebellar_hemorrhage': [0, 1],
    'ct_finding_cerebral_contusion':    [0, 1],
    'ct_finding_cerebral_edema':        [0, 1],
    'ct_finding_cerebral_hemorrhage':   [0, 1],
    'ct_finding_skull_diastasis':       [0, 1],
    'ct_finding_epidural_hematoma':     [0, 1],
    'ct_finding_extra_axial_hematoma':  [0, 1],
    'ct_finding_intraventricular_hemorrhage': [0, 1],
    'ct_finding_midline_shift':         [0, 1],
    'ct_finding_pneumocephalus':        [0, 1],
    'ct_finding_skull_fracture':        [0, 1],
    'ct_finding_subarachnoid_hemorrhage': [0, 1],
    'ct_finding_subdural_hematoma':     [0, 1],
    'ct_finding_traumatic_infarction':  [0, 1],
    'ct_extra_finding_diffuse_axonal_injury': [0, 1],
    'ct_extra_finding_herniation':      [0, 1],
    'ct_extra_finding_shear_injury':    [0, 1],
    'ct_extra_finding_sigmoid_sinus_thrombosis': [0, 1],
    'death_due_to_tbi':                 [0, 1],
    'hospitalized_2plus_nights_head_injury': [0, 1],
    'hospitalized_2plus_nights_with_positive_ct': [0, 1],
    'intubated_24plus_hours_head_trauma': [0, 1],
    'neurosurgery_performed':           [0, 1],
    'has_clinically_important_tbi':     [0, 1],
}

# Check for invalid values in each column
def check_invalid_values(df, valid_values):
    for col, valid in valid_values.items():
        invalid_mask = df[col].notna() & ~df[col].isin(valid)
        count = invalid_mask.sum()
        if count > 0:
            print(f"{col}: {count} invalid values")
            print(f"  Invalid values found: {sorted(df.loc[invalid_mask, col].unique())}")
            print()

check_invalid_values(tbi_data_df, valid_values)

### gcs_eye_score cannot take values of 5 (highest it goes is 4)
### gcs_verbal_score cannot take values of 6 (highest it goes is 6) and it also can't take 0 (none = 1)

- Investigate what happened, was this a missing value I imputed and thus more is incorrect? 
- from above the rows I fixed (imputed the missing component value to align with the sum - gcs_total_score was) : row 12931 and row 24761 

In [ ]:
invalid_gcs = tbi_data_df[
    ~tbi_data_df['gcs_eye_score'].isin([1, 2, 3, 4]) & tbi_data_df['gcs_eye_score'].notna() |
    ~tbi_data_df['gcs_verbal_score'].isin([1, 2, 3, 4, 5]) & tbi_data_df['gcs_verbal_score'].notna()
]

print(invalid_gcs[['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

### these are not the rows I imputed so these scores are mistakes
- can I fix them based on the other components and the final sum (that I trust more than the components?)

- easy fix is row 22282 gcs_verbal_score = 0 probably meant "none" which in the data documentation should be represented by 1 not 0 and this adjusting the total score by 1 (10->11) will not change the gcs_category group this row falls into 

- because the gcs_eye score for the other problem gcs_verbal row is okay and vice versa for the gcs_eye_score problem rows, my judgment call is supported and I can just go ahead and fix those values similarly to at the beginning. 
    - the only change now is these sums do match the total but I think likely the rows for gcs_eye_score were meant as the "most severe" which is 4 and not 5. So I think I can fix this here and adjust the gcs_total_score down by 1, (14 vs 15 is the same gcs_category so this won't really matter)

    - similarly for gcs_verbal_score = 6 probably meant the largest value it could take which is 5 and not 6 so this is a fair fix and adjusting the gcs_total_score down by 1 (15 -> 14) won't change the gcs_category this row falls into 

### APPLY THESE FIXES (data cleaning action item)

In [ ]:
# Fix gcs_eye_score = 5 → 4 (max valid value, rows 566 and 8583)
tbi_data_df.loc[tbi_data_df['gcs_eye_score'] == 5, 'gcs_total_score'] -= 1
tbi_data_df.loc[tbi_data_df['gcs_eye_score'] == 5, 'gcs_eye_score'] = 4

# Fix gcs_verbal_score = 0 → 1 (min valid value "none", row 22282)
tbi_data_df.loc[tbi_data_df['gcs_verbal_score'] == 0, 'gcs_total_score'] += 1
tbi_data_df.loc[tbi_data_df['gcs_verbal_score'] == 0, 'gcs_verbal_score'] = 1

# Fix gcs_verbal_score = 6 → 5 (max valid value, row 39855)
tbi_data_df.loc[tbi_data_df['gcs_verbal_score'] == 6, 'gcs_total_score'] -= 1
tbi_data_df.loc[tbi_data_df['gcs_verbal_score'] == 6, 'gcs_verbal_score'] = 5

# Verify
print(tbi_data_df.loc[[566, 8583, 22282, 39855], ['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score', 'gcs_total_score']])

### Another thing I noticed is patient_age_months had some 91's 
- because this was treated as a missing value for a different variable, I want to make sure that isn't happening here

In [ ]:
print(tbi_data_df[tbi_data_df['patient_age_months'] == 91][['patient_age_months', 'patient_age_years', 'patient_age_under_2yr']])

### more investigation, 91 could be months for a 7 year old but this many is sus 
- see if there is a consistent relationship (*12) between age in year and age in months for all but these rows (if so these rows are a problem)

In [ ]:
# Check how many rows have consistent month/year relationship
has_both = tbi_data_df['patient_age_months'].notna() & tbi_data_df['patient_age_years'].notna()

consistent = (tbi_data_df.loc[has_both, 'patient_age_years'] == 
              (tbi_data_df.loc[has_both, 'patient_age_months'] // 12).astype(int))

print(f"Rows with both age values: {has_both.sum()}")
print(f"Consistent rows: {consistent.sum()}")
print(f"Inconsistent rows: {(~consistent).sum()}")
print(f"Consistency rate: {consistent.mean():.1%}")

# Show the inconsistent ones
inconsistent_age = tbi_data_df[has_both & ~consistent][['patient_age_months', 'patient_age_years', 'patient_age_under_2yr']]
print("\nSample of inconsistent rows:")
print(inconsistent_age.head(20))

In [ ]:
# Check if 91 is still present or has been converted to NaN
print(f"Rows with patient_age_months = 91: {(tbi_data_df['patient_age_months'] == 91).sum()}")
print(f"Rows with patient_age_months = NaN: {tbi_data_df['patient_age_months'].isna().sum()}")
print(f"Total rows missing from consistency check: {43399 - has_both.sum()}")

In [ ]:
# Check specifically the 91 month rows
rows_91 = tbi_data_df[tbi_data_df['patient_age_months'] == 91]
print(rows_91[['patient_age_months', 'patient_age_years', 'patient_age_under_2yr']].head(10))

# Manually check: 91 // 12 = 7, so patient_age_years should be 7
print(f"\n91 // 12 = {91 // 12}")

### So this tells me I can leave it alone, its just rounding to 7 years so these 91s are fine. This also tells me I should leave the age in months missing if they are and not just multiply by 12 because floor division matches meaning that there are multiple different options for how many months that translate to a specific age in years 

# <span style="color: red;"> NEXT UP EDA: summary statistics (for cleaning - does it all make sense?) AND visualizations also for cleaning</span>

## Single variable visuals 
### Suggested figures for categorical variables 
- bar plot 

### for numeric variables 
- histogram 
- box plot


## Multiple variable visuals 
#### One categorical , one numeric 
- boxplots 
- barplot 

### one numeric, one time, one categorical 
- multiline plot 

In [ ]:
tbi_data_df.dtypes

### gcs_total_score is numeric, lets try a visual as EDA for cleaning 
- goal: should make sense with the documention 
    - inclusion/exclusion criteria 

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
tbi_data_df['gcs_total_score'].hist(bins=13, ax=ax1, edgecolor='black')
ax1.set_title('GCS Total Score Distribution')
ax1.set_xlabel('GCS Total Score')
ax1.set_ylabel('Count')
ax1.axvline(x=14, color='red', linestyle='--', label='Score = 14')
ax1.legend()

# Boxplot
tbi_data_df.boxplot(column='gcs_total_score', ax=ax2)
ax2.set_title('GCS Total Score Boxplot')
ax2.set_ylabel('GCS Total Score')

plt.tight_layout()
plt.show()

# Quick summary stats
print(tbi_data_df['gcs_total_score'].describe())
print(f"\nValue counts:\n{tbi_data_df['gcs_total_score'].value_counts().sort_index()}")

### JUDGMENT CALL: 
Looking at the article's flowchart, it explicitly says "969 patients with GCS 3-13 excluded" - but excluded from the prediction rule derivation, not necessarily from the dataset itself. The documentation notes "Patients with ventricular shunts, bleeding disorders, and GCS scores less than 14 were enrolled but are being analysed separately" - meaning they were intentionally collected, just for a different analysis.
So dropping them would be throwing away real data that was deliberately enrolled. The better judgment call is to flag them:

<span style="color: red;">maybe a preprocessing task to drop them?</span>


In [ ]:
# Create a flag column for GCS exclusion criteria
tbi_data_df['in_primary_analysis'] = (tbi_data_df['gcs_total_score'] >= 14).astype(int)

# Verify
print(tbi_data_df['in_primary_analysis'].value_counts())
print(f"\nGCS 14-15 (in primary analysis): {(tbi_data_df['gcs_total_score'] >= 14).sum()}")
print(f"GCS 3-13 (separate analysis): {(tbi_data_df['gcs_total_score'] < 14).sum()}")

#### Last two numeric columns to investigate: patient_age_months ; patient_age_years

Visualizations make sense here because:

- Age has a meaningful distribution shape you'd want to see - is it roughly uniform across 0-17 years or skewed toward certain ages?
    - Boxplots will show if there are any outlier ages that seem impossible (e.g., someone recorded as 200 months old)
    - The article split the analysis at 2 years old, so seeing how many patients fall below/above that cutoff visually is useful

- Numeric summaries also make sense because:

    - Mean/median age tells you about the typical patient in the study
    - We already know the valid range (0-17 years, 0-215 months) so min/max will confirm no out-of-range values slipped through
    - Checking for the 91-month issue we found earlier is easier with value counts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- patient_age_years ---
# Histogram/bar chart
tbi_data_df['patient_age_years'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,0], edgecolor='black', color='steelblue'
)
axes[0,0].set_title('Patient Age (Years) Distribution')
axes[0,0].set_xlabel('Age (Years)')
axes[0,0].set_ylabel('Count')
axes[0,0].axvline(x=2, color='red', linestyle='--', label='2 year cutoff')
axes[0,0].legend()
axes[1,0].set_xticks(range(0, 18, 1))

# Boxplot
tbi_data_df.boxplot(column='patient_age_years', ax=axes[0,1])
axes[0,1].set_title('Patient Age (Years) Boxplot')
axes[0,1].set_ylabel('Age (Years)')
axes[1,1].set_yticks(range(0, 18, 1))


# --- patient_age_months ---
# Histogram
tbi_data_df['patient_age_months'].plot(
    kind='hist', bins=30, ax=axes[1,0], edgecolor='black', color='steelblue'
)
axes[1,0].set_title('Patient Age (Months) Distribution')
axes[1,0].set_xlabel('Age (Months)')
axes[1,0].set_ylabel('Count')
axes[1,0].axvline(x=24, color='red', linestyle='--', label='24 month cutoff')
axes[1,0].legend()
axes[1,0].set_xticks(range(0, 221, 20))

# Boxplot
tbi_data_df.boxplot(column='patient_age_months', ax=axes[1,1])
axes[1,1].set_title('Patient Age (Months) Boxplot')
axes[1,1].set_ylabel('Age (Months)')
axes[1,1].set_yticks(range(0, 221, 50))
axes[1,1].set_yticks(list(axes[1,1].get_yticks()) + [215])

plt.tight_layout()
plt.savefig('age_distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Numeric summaries
print("=== patient_age_years ===")
print(tbi_data_df['patient_age_years'].describe())
print(f"\nUnder 2 years: {(tbi_data_df['patient_age_years'] < 2).sum()}")
print(f"2 years and older: {(tbi_data_df['patient_age_years'] >= 2).sum()}")

print("\n=== patient_age_months ===")
print(tbi_data_df['patient_age_months'].describe())
print(f"\nMissing: {tbi_data_df['patient_age_months'].isna().sum()}")
print(f"Max valid (17 years = 215 months): {(tbi_data_df['patient_age_months'] > 215).sum()} rows exceed this")

### Distribution shape:

- Both age distributions are right-skewed (younger patients dominate), which makes clinical sense - young children are more likely to suffer head trauma from falls and accidents
- The spike at ages 0-1 in years (and 0-24 months) reflects the study's particular interest in very young children who are most vulnerable to radiation from CT scans, which was a key motivation of the study

- The 2-year cutoff:
    - You can clearly see the split the article used - 10,904 patients under 2 years vs 32,495 aged 2 and older, roughly a 25/75 split which matches exactly what the article reported (25% under 2 years)

- Data quality findings:

    - patient_age_years has no missing values (count = 43,399) while patient_age_months has 136 missing - these are the ones we couldn't impute earlier because we didn't want to guess months for older children, which was the right call
    - Max age in months is 215 (exactly 17 years 11 months) and min is 0, so no out-of-range values exist
    - The standard deviation for months (66.6) being very large relative to the mean (84.4) confirms the wide spread visible in the histogram, which is expected given the 0-17 year range

Overall: the age data looks clean and consistent with the study's enrolled population!

# Explore missing values 

In [ ]:
missing_pct = (tbi_data_df.isnull().sum() / len(tbi_data_df) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

plt.figure(figsize=(50, 25))
missing_pct.plot(kind='barh', color='steelblue')
plt.title('Percentage Missing by Column')
plt.xlabel('% Missing')

plt.tight_layout()
plt.savefig('missing_data_percentage.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Outcome variable distributions - key for understanding the data

In [ ]:
primary = tbi_data_df[tbi_data_df['in_primary_analysis'] == 1]

outcome_cols = [
    'has_clinically_important_tbi', 'ct_shows_tbi', 
    'death_due_to_tbi', 'neurosurgery_performed',
    'hospitalized_2plus_nights_head_injury',
    'intubated_24plus_hours_head_trauma'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), outcome_cols):
    (primary[col].value_counts(dropna=False, normalize=True) * 100).plot(kind='bar', ax=ax, edgecolor='black')
    for p in ax.patches: ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=9)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Percentage (%)')
plt.tight_layout()
plt.savefig('outcome_var_distributions.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Overall pattern: all outcome variables are heavily skewed toward 0 (no outcome), which is exactly what you'd expect for a minor head trauma population.
- has_clinically_important_tbi - Vast majority are 0, with a very small proportion being 1. This matches our earlier sanity check of ~0.9% ciTBI rate. Small NaN bar is the 20 missing values we noted earlier - acceptable.

- ct_shows_tbi - This one is notably different - the largest bar is actually NaN, not 0. This makes sense because ~63% of patients never got a CT scan at all (no CT = not applicable = NaN). Of those who did get CT, most were negative (0) with a small positive (1) group.

- death_due_to_tbi - Extremely rare as expected, almost entirely 0. The tiny NaN bar (8 missing values) is negligible.

- neurosurgery_performed - Extremely rare, consistent with the article reporting ~0.1% neurosurgery rate. Looks clean.

- hospitalized_2plus_nights_head_injury - Overwhelmingly 0 (sent home), with a small proportion admitted. Consistent with the study population being minor head trauma.

- intubated_24plus_hours_head_trauma - Extremely rare, as expected for GCS 14-15 patients.

- From a cleaning standpoint: nothing concerning here. All distributions are clinically logical for a minor pediatric head trauma study, and the NaN patterns are all explainable!

### GCS component distributions - verify they look right

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['gcs_eye_score', 'gcs_verbal_score', 'gcs_motor_score']):
    tbi_data_df[col].value_counts().sort_index().plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('gcs_component_distribution.png', dpi=300, bbox_inches = 'tight', facecolor='white')
plt.show()



### All three plots show the same pattern: an overwhelming majority of patients scored at the maximum value for each component:
- gcs_eye_score: nearly all scored 4 (Spontaneous eye opening - best possible)
- gcs_verbal_score: nearly all scored 5 (Oriented - best possible)
- gcs_motor_score: nearly all scored 6 (Follows commands - best possible)
    - This is completely expected and supports data quality because:

- Consistent with inclusion criteria - the study enrolled patients with GCS 14-15, meaning only minor head trauma. 
    - We'd expect most patients to be at or near maximum scores on all components since 4+5+6 = 15 (perfect GCS)
- No unexpected values remain - after our earlier cleaning fixes (correcting the eye score of 5, verbal score of 0 and 6), the only values present are within the valid ranges we defined. The distributions confirm those fixes worked.
- The small bars at lower scores make clinical sense - the handful of patients with lower component scores correspond to the GCS 14 patients (about 1,346 of them from our earlier value counts), who by definition have at least one component below maximum

- From a cleaning standpoint: nothing concerning here! The distributions look exactly as we'd expect for a minor pediatric head trauma dataset.

### ciTBI rate by age group - (sanity check against article)

In [ ]:
# Convert to numeric just for this calculation
citbi_numeric = tbi_data_df['has_clinically_important_tbi'].astype(float)
age_group = tbi_data_df['patient_age_under_2yr']

# Calculate ciTBI rate per age group
citbi_by_age = citbi_numeric.groupby(age_group).mean() * 100

fig, ax = plt.subplots(figsize=(6, 4))
citbi_by_age.plot(kind='bar', ax=ax, edgecolor='black', color='steelblue')
ax.set_title('ciTBI Rate by Age Group')
ax.set_xlabel('Age Group (1 = Under 2yr, 2 = 2yr and older)')
ax.set_ylabel('ciTBI Rate (%)')
ax.axhline(y=0.9, color='red', linestyle='--', label='Expected rate (0.9%)')
ax.legend()
plt.tight_layout()
plt.show()

# Print exact rates
print("ciTBI rates by age group:")
print(citbi_by_age)
print("\nArticle reported: ~0.9% for both age groups")

Both rates are higher than the article's reported 0.9% - you're seeing ~1.4% for under 2 years and ~1.9% for 2 years and older.
- This is actually expected and explainable for a few reasons:
Why your rates are higher:

- Your dataset includes GCS 3-13 patients - remember those 969 rows we flagged with in_primary_analysis = 0? Those patients have much higher ciTBI rates since they have worse neurological status. The article's 0.9% was calculated only on the GCS 14-15 population
    - The article's numbers were from the derivation + validation populations only (42,412 patients) while your dataset has 43,399 rows total - the extra ~1,000 rows are those excluded patients

### Quick Sanity check - filter to just the primary analysis population

In [ ]:
citbi_numeric = tbi_data_df['has_clinically_important_tbi'].astype(float)
age_group = tbi_data_df['patient_age_under_2yr']

# Filter to GCS 14-15 only
primary = tbi_data_df['in_primary_analysis'] == 1

citbi_by_age = citbi_numeric[primary].groupby(age_group[primary]).mean() * 100
print(citbi_by_age)

- sanity check passes!
    - Under 2 years: 0.91% (article reported 0.86-1.13%)
    - 2 years and older: 0.88% (article reported 0.85-0.98%)

- Both rates are essentially spot on with the article's reported ~0.9% for both age groups. This is strong evidence that data cleaning has been done correctly - I haven't accidentally introduced errors or dropped important rows. The data matches the published results from the original study.

### Categorical variable distributions - check for unexpected imbalances

In [ ]:
primary = tbi_data_df[tbi_data_df['in_primary_analysis'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ['injury_mechanism', 'injury_mechanism_severity']):
    (primary[col].value_counts(dropna=False, normalize=True) * 100).sort_index().plot(
        kind='bar', ax=ax, edgecolor='black'
    )
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%', 
                    (p.get_x() + p.get_width()/2, p.get_height()), 
                    ha='center', va='bottom', fontsize=9)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Percentage (%)')

plt.tight_layout()
plt.savefig('mechanism_distributions.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
cat_cols = [
    'injury_mechanism', 'injury_mechanism_severity',
    'patient_gender', 'patient_race', 'patient_ethnicity',
    'ed_discharge_status', 'gcs_category'
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), cat_cols):
    tbi_data_df[col].value_counts(dropna=False).plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

#### What the plots tell me:
- injury_mechanism - Fall from elevation (8) is the most common, followed by sports (6) and MVC (1). This makes clinical sense for a pediatric head trauma study. The distribution looks reasonable.
- injury_mechanism_severity - Moderate (2) dominates heavily (~30,000 rows), with low (1) and high (3) being much less common. - This aligns with the article which reported ~70% moderate mechanisms.

- patient_gender - Males (1) outnumber females (2) roughly 60/40. Expected - males tend to have higher rates of head trauma.
- patient_race - Heavily skewed toward White (1) and Black (2), with very few other races. Worth noting as a limitation of the study's generalizability.

- patient_ethnicity - Notably has a large NaN bar - worth investigating how many are missing ethnicity data.
- ed_discharge_status - Overwhelmingly home (1) which makes sense given the study enrolled minor head trauma patients with GCS 14-15.
- gcs_category - Almost entirely category 2 (GCS 14-15) with a tiny sliver of category 1 (GCS 3-13) - exactly what we'd expect given our in_primary_analysis flag discussion

- Nothing here screams data quality issue - these all look clinically reasonable!

### <span style="color: red;">Next idea: some of the variables have 92 as NA where maybe 0 would be better than treating as NaN (missing) - come back to this</span>

# Support for my judgment call: 

The case for keeping as NaN is strong here:
The 92 "not applicable" values are fundamentally different from truly missing data, but in practice NaN is still the right choice because:

Structurally they ARE missing - if has_posttraumatic_seizure = 0, then seizure_duration has no meaningful value to impute. There's no "none" duration - the question simply doesn't apply. NaN correctly communicates "this field has no value for this patient"
Imputing a new category would be misleading - if you created a value like 0 or -1 for "not applicable", any future analysis would need to carefully exclude it. NaN is automatically excluded from most statistical operations in pandas, which is exactly the behavior you want for these fields
The information isn't lost - the "not applicable" information is already captured by the parent variable. If has_posttraumatic_seizure = 0, you already KNOW seizure_duration is not applicable. You don't need to store that information twice.
Standard practice - in clinical datasets, "not applicable" sub-variables are conventionally left as NaN when the parent condition is absent. Any analyst familiar with this type of data would expect this.

The only case where I'd want to distinguish "not applicable" from "truly missing" would be if you needed to know whether a field was skipped intentionally vs accidentally - but your conditional validation checks earlier already confirmed these patterns are consistent!



# <span style = "color:red;">Question</span>
- variables that are almost entirely missing , exclude? 
- which rows to exclude or not (lots of missing?)

## Answer to these questions (my support for judgment calls I make): 
- Should we exclude almost-entirely-missing columns?
    - Looking at the chart, the columns with 95-99% missing are all the "not applicable" sub-detail variables like skull_fracture_depressed, has_basilar_*, seizure_timing, ct_sedation_reason_* etc. I would keep them because:

- The missingness is structured and meaningful, not random - they're only filled in for the rare patients who had that condition
    - They could be analytically important - e.g. skull_fracture_depressed only applies to ~1% of patients but is clinically very significant
    - Dropping them loses real clinical information

- Should we exclude rows based on missing data?
    - I'd say no for most rows - same reasoning, missingness is structured not random. However there are specific rows worth flagging or excluding based on the article's explicit exclusion criteria:

- GCS < 14 - we already flagged these with in_primary_analysis
    - Penetrating trauma - the article excluded these but we have no variable to identify them
    - Known brain tumors or pre-existing neurological disorders - same issue, no variable
    - Neuroimaging at outside hospital before transfer - no variable for this either
    - Trivial mechanisms with no symptoms - partially captured by injury_mechanism_severity

Final thoughts: I will not drop any rows or columns at this stage. Instead keep the in_primary_analysis flag created and let downstream analyses filter as needed. 
    - The data was intentionally collected this way and removing variables or rows could compromise the integrity of future analyses.


# <span style="color: red;">last thing forgot to look into : OUTLIERS</span>

- checked for outliers in age (cell 149-151) and explicitly concluded there were none — the distributions looked exactly as expected for a pediatric study, right-skewed toward younger ages, max age 17, no impossible values like 200 months. The boxplots you made confirmed this. So no outlier cleaning action is needed or appropriate for age.

- For the other numeric columns — gcs_total_score (range 3-15), gcs_eye_score (1-4), gcs_verbal_score (1-5), gcs_motor_score (1-6), patient_number — these are all bounded categorical or near-categorical values with documented valid ranges. You already handle out-of-range values in fix_gcs_entry_errors() and validate_data() Check 1. There are no continuous unbounded variables where a statistical outlier definition (e.g., IQR-based) would be meaningful.

- So for this dataset, "handling outliers" is already covered by your valid value range checks — that's the appropriate approach when variables are clinically defined with fixed ranges. What you'd write in your report is something like: "Given that all numeric variables are either bounded by clinical definitions (GCS scores) or constrained by study design (age 0-17), we investigated outliers by checking for values outside documented valid ranges rather than applying statistical outlier detection. No implausible values were found after cleaning." The ciTBI sanity check in cell 165 — where your cleaned rates matched the article's published figures exactly — is also strong evidence your cleaning didn't introduce any distortions and serves as your reality check.

## Check for duplicate rows (after cleaning) - just thought of this 

In [ ]:
import pandas as pd
import sys

sys.path.append("../../code")
from clean import convert_data_types

cleaned = pd.read_csv("../../data/TBI_PUD_cleaned.csv")
cleaned = convert_data_types(cleaned)

# Check 1: Duplicate rows (exact same values across ALL columns)
duplicate_rows = cleaned.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows}")

# Check 2: Duplicate patient numbers (same patient appearing twice)
duplicate_patients = cleaned.duplicated(subset=['patient_number']).sum()
print(f"Duplicate patient_number values: {duplicate_patients}")

# Check 3: If any duplicates found, show them
if duplicate_patients > 0:
    dupes = cleaned[cleaned.duplicated(subset=['patient_number'], keep=False)]
    print("\nRows with duplicate patient numbers:")
    print(dupes[['patient_number', 'patient_age_years', 'patient_gender', 'injury_mechanism']].sort_values('patient_number'))
else:
    print("\nAll patient numbers are unique — one row per patient ✓")

# Check 4: Confirm total rows matches expected
print(f"\nTotal rows: {len(cleaned):,} (expected 43,399)")
print(f"Unique patient numbers: {cleaned['patient_number'].nunique():,}")

In [ ]:
cleaned.dtypes